# v0.19 Audit-Corrected Research & Fixed-Window Forward Microstructure
This notebook reproduces the v0.19 audit and public REST microstructure smoke test. It does **not** require private exchange credentials and cannot enable real-money trading.

The primary thesis forecast/trading horizon remains **4h**. The collector uses a **60-second point-in-time reported-trade window** and targets **30-minute measurement cadence** only to improve within-bar measurement quality.

In [ ]:
!rm -rf modular-crypto-trading-bot
!git clone https://github.com/parsa314/modular-crypto-trading-bot.git
%cd modular-crypto-trading-bot
!git fetch --all
!git checkout v19-audit-corrected-forward-microstructure


In [ ]:
!python -m pip install -q --upgrade pip
!pip install -q -e '.[dev]'


## 1) Deterministic audit, PIT-window and regression tests

In [ ]:
!pytest -q tests/test_audit_v19.py tests/test_forward_microstructure_v19.py tests/test_v19_collector_helpers.py tests/test_forward_paper_v14.py tests/test_cost_regime_v18.py tests/test_evidence_ledger_v17.py tests/test_trial_registry_v17.py


## 2) Audit-corrected historical reconstruction (non-promotional)
The original v0.18 raw OHLCV bytes were not archived, so this is a refreshed reconstruction audit and cannot replace the immutable v0.18 result.

In [ ]:
!python scripts/run_v19_audit.py --bars 1800 --output artifacts/v19/v19_audit.json --markdown artifacts/v19/V19_AUDIT_RESULTS.md


In [ ]:
import json
from pathlib import Path
from research_bot.integrity_v19 import payload_sha256
audit = json.loads(Path('artifacts/v19/v19_audit.json').read_text())
print('manifest:', audit['manifest_sha256'])
print('hash valid:', audit['manifest_sha256'] == payload_sha256(audit, exclude_keys=('manifest_sha256',)))
print('cost audit status:', audit['cost_audit']['evidence_status'])
print('regime audit status:', audit['regime_audit']['evidence_status'])
print('live authorized:', audit['live_execution_authorized'])


## 3) Prospective multi-venue fixed-window REST snapshot
Trade rows are locally restricted to `[book_time-60s, book_time]`. Provider-returned future rows are excluded. Public trade `side` is treated as exchange-reported and not independently verified aggressor ground truth.

In [ ]:
!python scripts/collect_v19_forward_microstructure.py --trade-window-seconds 60 --output artifacts/v19/forward_microstructure_snapshot.json


In [ ]:
snap = json.loads(Path('artifacts/v19/forward_microstructure_snapshot.json').read_text())
print('snapshot:', snap['snapshot_sha256'])
print('hash valid:', snap['snapshot_sha256'] == payload_sha256(snap, exclude_keys=('snapshot_sha256',)))
print('measurement contract:', snap['measurement_contract'])
print('authorized symbols:', snap['authorized_symbol_count'])
print('signal authorized:', snap['signal_authorized'])
for row in snap['symbols']:
    print(row['symbol'], row['status'], 'venues=', row.get('accepted_venues'), 'reported_flow=', row.get('mean_reported_trade_imbalance'), 'clock_skew_s=', row.get('venue_clock_skew_seconds'))
    for venue in row.get('venues', []):
        print('  ', venue['venue'], 'recent=', venue['trade_count'], 'staleness_s=', venue['trade_staleness_seconds'], 'future_excluded=', venue['future_trade_count_excluded'])


## 4) Scientific interpretation
- v0.18 uncertainty calibration is audited with expanding OOF residuals and terminal closing costs.
- v0.18-B portfolio/regime geometry is reconstructed to match v0.11, but remains retrospective audit evidence.
- v0.19 removes the variable latest-N trade-window bug by enforcing a 60-second PIT window.
- the 30-minute schedule is a **measurement cadence**, not a new trading/target horizon.
- public REST snapshots are not a complete L2/L3 event stream.
- no current output authorizes PAPER strategy replacement, testnet or LIVE execution.